# QFF 2026 QML Q1 — reproducible synthetic study

**Selected working problem statement:** classify a four-feature synthetic binary benchmark with a four-qubit VQC, compare it with logistic regression, and simulate the original VQC on illustrative five- and nine-qubit processor maps. The repository contains no authoritative organizer specification; dataset and coupling-map authority remain unresolved in `docs/DECISIONS.md`. Synthetic label 1 is a positive-class proxy, not a medical diagnosis.

Run all cells from the repository root after installing `requirements.txt` in the notebook kernel. Outputs are written to separate `results/notebook_*` files so the recorded baseline artifacts are preserved. Execution includes four bounded VQC fits and Aer simulations; it does not download MIT-BIH data.

In [ ]:
from pathlib import Path
import json
import numpy as np
from src.datasets import make_synthetic_benchmark
from src.baseline import run_synthetic_baseline
from src.vqc import run_synthetic_vqc
from src.vqc_experiments import run_controlled_experiments
from src.processor_benchmark import run_benchmark
from src.result_figures import render_result_figures
root = Path.cwd()
assert (root / 'requirements.txt').is_file(), 'Launch the notebook from the repository root'
out = root / 'results'
out.mkdir(exist_ok=True)

## Data and method

The benchmark is generated by scikit-learn `make_classification` with 200 rows, four features, and seed 42. A stratified seed-42 split yields 160 training and 40 held-out test rows. The shared `MinMaxScaler` learns only from training rows and maps angles to `[0, π]`. For VQC selection, a separate stratified 128/32 fit/validation split is made *inside* the 160 training rows; its scaler is fit only on those 128 fit rows. The 40 test labels play no part in variant selection.

In [ ]:
dataset = make_synthetic_benchmark()
print('train/test shapes:', dataset.X_train.shape, dataset.X_test.shape)
print('train/test label counts:', np.bincount(dataset.y_train), np.bincount(dataset.y_test))
print('scaler range:', dataset.scaler.feature_range)

## Classical baseline and original VQC

Logistic regression uses the shared arrays with seed 42 and `max_iter=1000`. The original quantum model uses RY angle encoding, two RY/RZ plus ring-CX layers, parity readout, exact QMLSampler statevector probabilities, and COBYLA limited to 25 evaluations.

In [ ]:
baseline = run_synthetic_baseline(out / 'notebook_logistic_regression.json')
original = run_synthetic_vqc(out / 'notebook_original_vqc.json')
print('Logistic held-out:', {k: getattr(baseline, k) for k in ('accuracy', 'abnormal_recall', 'precision', 'f1')})
print('Original VQC held-out accuracy:', original.test_accuracy)

## Bounded VQC selection

Compare exactly three prespecified variants: original ring/parity, line/parity to reduce routing, and ring/first-qubit readout. All use identical initial-weight seed 42 and 25 COBYLA evaluations. Rank by validation label-1 recall, then precision, F1, lower A/B routing CX overhead, and listed order. Only after selection, refit the chosen architecture on all 160 training rows and evaluate the 40 held-out rows once.

In [ ]:
experiment = run_controlled_experiments(
    experiment_path=out / 'notebook_vqc_controlled_experiments.json',
    selected_path=out / 'notebook_vqc_selected_parameters.json',
)
for candidate in experiment['candidates']:
    print(candidate['variant']['name'], candidate['validation']['recall_label_1'], candidate['validation']['precision_label_1'], candidate['validation']['f1_label_1'])
print('Selected:', experiment['selected_variant_name'])
print('Selected held-out:', experiment['final_refit_and_held_out']['metrics'])

## Processor A/B simulation of the **original** VQC

This comparison reuses one saved original ring/parity weight vector and the same held-out rows. A is a custom bidirectional five-qubit line; B is a custom bidirectional nine-qubit 3×3 grid. Both use `rz/sx/x/cx`, transpiler level 1, seed 42, 4,096 shots, and uniform illustrative depolarizing/readout error rates. These maps and rates are not organizer reference devices or calibrations. This is separate from the selected first-qubit-readout VQC.

In [ ]:
processor = run_benchmark(
    parameters_path=out / 'notebook_original_vqc.json',
    output_path=out / 'notebook_processor_benchmark.json',
)
for item in processor['processors']:
    print(item['name'], 'CX', item['cx_count'], 'depth', item['depth'], 'ideal/noisy recall', item['ideal']['recall_label_1'], item['noisy']['recall_label_1'])

## Figures and interpretation

These SVGs read only the notebook-generated JSON files. The selected VQC showed higher recall on the 32-row inner validation set but much lower recall on the 40-row held-out set. The logistic baseline remains stronger on the recorded held-out benchmark. See `report.md` for metrics, limitations, and provenance.

In [ ]:
figures = render_result_figures(
    baseline_path=out / 'notebook_logistic_regression.json',
    experiment_path=out / 'notebook_vqc_controlled_experiments.json',
    processor_path=out / 'notebook_processor_benchmark.json',
    output_dir=out / 'notebook_figures',
)
print('Figures:', *figures, sep='\n')